<a href="https://colab.research.google.com/github/Nik2416/FastAPI-/blob/main/Insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.pipeline import Pipeline

In [ ]:
df=pd.read_csv('insurance.csv')

In [ ]:
df

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92000,False,Jaipur,retired,High
1,36,101.1,1.83,34.28000,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64000,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34000,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94000,True,Indore,retired,High
...,...,...,...,...,...,...,...,...
95,36,52.8,1.57,19.64000,False,Indore,business_owner,Low
96,26,113.8,1.54,34.01000,False,Delhi,private_job,Low
97,52,60.8,1.80,44.86000,False,Hyderabad,freelancer,Low
98,27,101.1,1.82,28.30000,False,Kolkata,business_owner,Low


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         100 non-null    int64  
 1   weight                      100 non-null    float64
 2   height                      100 non-null    float64
 3   income_lpa                  100 non-null    float64
 4   smoker                      100 non-null    bool   
 5   city                        100 non-null    object 
 6   occupation                  100 non-null    object 
 7   insurance_premium_category  100 non-null    object 
dtypes: bool(1), float64(3), int64(1), object(3)
memory usage: 5.7+ KB


In [ ]:
df.describe()

,age,weight,height,income_lpa
count,100.000000,100.000000,100.000000,100.000000
mean,47.180000,83.894000,1.713200,18.400600
std,16.649312,21.020278,0.110205,16.067465
min,18.000000,51.100000,1.500000,0.530000
25%,34.750000,63.650000,1.610000,2.897500
50%,47.000000,82.300000,1.730000,14.122583
75%,61.000000,101.300000,1.810000,30.162500
max,75.000000,119.800000,1.900000,50.000000


In [ ]:
df.isnull().sum()

,0
age,0
weight,0
height,0
income_lpa,0
smoker,0
city,0
occupation,0
insurance_premium_category,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.shape

(100, 8)

In [ ]:
df.corr(numeric_only=True)

,age,weight,height,income_lpa,smoker
age,1.000000,0.033913,-0.066323,-0.163108,0.042993
weight,0.033913,1.000000,0.108447,-0.113350,0.014156
height,-0.066323,0.108447,1.000000,0.099109,-0.093504
income_lpa,-0.163108,-0.113350,0.099109,1.000000,0.023104
smoker,0.042993,0.014156,-0.093504,0.023104,1.000000


In [ ]:
df['city'].unique()

array(['Jaipur', 'Chennai', 'Indore', 'Mumbai', 'Kota', 'Hyderabad',
       'Delhi', 'Chandigarh', 'Pune', 'Kolkata', 'Lucknow', 'Gaya',
       'Jalandhar', 'Mysore', 'Bangalore'], dtype=object)

In [ ]:
df['insurance_premium_category'].unique()

array(['High', 'Low', 'Medium'], dtype=object)

In [ ]:
df['bmi']=df['weight']/(df['height']**2)

In [ ]:
def age_group(age):
  if age<25:
    return 'young'
  elif  age<45:
    return 'adult'
  elif age<60:
    return 'middle_age'
  return 'old'

In [ ]:
df['age_group']=df['age'].apply(age_group)

In [ ]:
def lifestyle(row):
  if row['smoker'] and row['bmi']>30:
    return 'high'
  elif row['smoker'] or row['bmi']<25:
    return 'medium'
  else:
    return 'low'

In [ ]:
df['lifestyle_risk']=df.apply(lifestyle,axis=1)

In [ ]:
tier_1= [
    "Ahmedabad", "Bengaluru", "Chennai", "Delhi",
    "Hyderabad", "Kolkata", "Mumbai", "Pune"
]
tier_2 = [
    "Agra", "Ajmer", "Amritsar", "Chhatrapati Sambhajinagar",
    "Bhopal", "Bhubaneswar", "Chandigarh", "Coimbatore", "Cuttack",
    "Dehradun", "Dhanbad", "Durg-Bhilai", "Faridabad", "Ghaziabad",
    "Guwahati", "Gwalior", "Indore", "Jaipur", "Jalandhar", "Jammu",
    "Jamnagar", "Jamshedpur", "Jodhpur", "Kanpur", "Kochi", "Kota",
    "Kozhikode", "Lucknow", "Ludhiana", "Madurai", "Mangaluru", "Meerut",
    "Mysuru", "Nagpur", "Nashik", "Patna", "Puducherry", "Raipur",
    "Rajkot", "Ranchi", "Salem", "Srinagar", "Surat", "Thiruvananthapuram",
    "Tiruchirappalli", "Udaipur", "Vadodara", "Varanasi", "Vijayawada",
    "Visakhapatnam"
]

In [ ]:
def city_tier(city):
  if city in tier_1:
    return 1
  elif city in tier_2:
    return 2
  else:
    return 3

In [ ]:
df['city_tier']=df['city'].apply(city_tier)

In [ ]:
df

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
0,67,119.8,1.56,2.92000,False,Jaipur,retired,High,49.227482,old,low,2
1,36,101.1,1.83,34.28000,False,Chennai,freelancer,Low,30.189017,adult,low,1
2,39,56.8,1.64,36.64000,False,Indore,freelancer,Low,21.118382,adult,medium,2
3,22,109.4,1.55,3.34000,True,Mumbai,student,Medium,45.535900,young,high,1
4,69,62.2,1.60,3.94000,True,Indore,retired,High,24.296875,old,medium,2
...,...,...,...,...,...,...,...,...,...,...,...,...
95,36,52.8,1.57,19.64000,False,Indore,business_owner,Low,21.420747,adult,medium,2
96,26,113.8,1.54,34.01000,False,Delhi,private_job,Low,47.984483,adult,low,1
97,52,60.8,1.80,44.86000,False,Hyderabad,freelancer,Low,18.765432,middle_age,medium,1
98,27,101.1,1.82,28.30000,False,Kolkata,business_owner,Low,30.521676,adult,low,1


In [ ]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [ ]:
df.drop(columns=['age','weight','height','smoker','city'])[['income_lpa','occupation','bmi','age_group','lifestyle_risk','city_tier','insurance_premium_category']]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,2.92000,retired,49.227482,old,low,2,High
1,34.28000,freelancer,30.189017,adult,low,1,Low
2,36.64000,freelancer,21.118382,adult,medium,2,Low
3,3.34000,student,45.535900,young,high,1,Medium
4,3.94000,retired,24.296875,old,medium,2,High
...,...,...,...,...,...,...,...
95,19.64000,business_owner,21.420747,adult,medium,2,Low
96,34.01000,private_job,47.984483,adult,low,1,Low
97,44.86000,freelancer,18.765432,middle_age,medium,1,Low
98,28.30000,business_owner,30.521676,adult,low,1,Low


In [ ]:
X=df[['bmi','age_group','lifestyle_risk','city_tier','occupation','income_lpa']]
y=df['insurance_premium_category']

In [ ]:
categorical=['age_group','lifestyle_risk','occupation','city_tier']
numerical=['bmi','income_lpa']

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',OneHotEncoder(),categorical),
        ('num','passthrough',numerical)
    ]
)

In [ ]:
pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier',RandomForestClassifier(random_state=42))
])

In [ ]:
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (80, 6)
y_train shape: (80,)


In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
y_pred= pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

0.8

In [ ]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,occupation,income_lpa
84,28.801497,old,medium,2,retired,0.62
39,35.643424,middle_age,high,3,unemployed,11.99
52,47.344720,young,low,2,student,2.96
51,38.827923,middle_age,high,2,private_job,28.95
56,42.414152,young,high,1,student,2.86


In [ ]:
import pickle

pickle_model='model.pkl'
with open(pickle_model,'wb') as f:
  pickle.dump(pipeline,f)